In [4]:
# =============================================================================
# CANADA-BRAZIL TRADE OPPORTUNITIES REPORT
# FINAL CONSOLIDATED SCRIPT
#
# OUTPUTS
# - reports/trade-report.html
# - reports/trade-report.pdf
# - reports/report_text/trade-report_text.md
#
# INCLUDED VISUALS
# - Q1  Annual Exports vs Imports
# - Q1b Monthly YoY comparison
# - Q2  Province Growth & Decline
# - Q3  Trade Balance monthly trend
# - Q4  HS Chapter Winners vs Risks
# - Q5  Provincial concentration / contribution
# - Q6  Baseline outlook
#
# NOTES
# - safer PDF layout
# - avoids most chart/table page splits
# - auto-creates editable report text file
# - handles common path / data variations
# =============================================================================

# =============================================================================
# 0. INSTALLS
# =============================================================================
import subprocess
import sys

REQUIRED_PACKAGES = [
    "jinja2",
    "weasyprint",
    "openpyxl",
    "markdown",
    "xhtml2pdf",
    "reportlab",
    "scikit-learn",
    "statsmodels",
    "Pillow"
]

for pkg in REQUIRED_PACKAGES:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    except Exception:
        pass

# =============================================================================
# 1. IMPORTS
# =============================================================================
import os
import io
import re
import math
import base64
import warnings
from io import BytesIO
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

try:
    from jinja2 import Template
    JINJA_AVAILABLE = True
except Exception:
    JINJA_AVAILABLE = False

try:
    from weasyprint import HTML, CSS
    from weasyprint.text.fonts import FontConfiguration
    WEASYPRINT_AVAILABLE = True
except Exception:
    WEASYPRINT_AVAILABLE = False

# =============================================================================
# 2. PATHS
# =============================================================================
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        print("✅ Google Drive mounted")
    except Exception as e:
        print(f"⚠️ Drive mount warning: {e}")

candidate_bases = [
    Path("/content/drive/MyDrive/canada-brazil-trade-report"),
    Path("/content/drive/MyDrive/business-report"),
    Path("/content/drive/MyDrive/data-analytics-projects/canada-brazil-trade-report"),
    Path.cwd(),
    Path.cwd().parent,
    Path("./canada-brazil-trade-report"),
    Path("./business-report"),
    Path("."),
]

BASE_DIR = None
for p in candidate_bases:
    p = Path(p)
    if p.exists():
        if (p / "data").exists() or (p / "reports").exists() or p.name.lower() in [
            "canada-brazil-trade-report", "business-report"
        ]:
            BASE_DIR = p.resolve()
            break

if BASE_DIR is None:
    BASE_DIR = Path(".").resolve()

if BASE_DIR.name.lower() == "notebooks":
    BASE_DIR = BASE_DIR.parent.resolve()

DATA_DIR = BASE_DIR / "data"
CLEAN_DIR = DATA_DIR / "clean"
RAW_DIR = DATA_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"
IMG_DIR = REPORTS_DIR / "report_images"
TEXT_DIR = REPORTS_DIR / "report_text"
NOTEBOOKS_DIR = BASE_DIR / "notebooks"

HTML_OUTPUT = REPORTS_DIR / "trade-report.html"
PDF_OUTPUT = REPORTS_DIR / "trade-report.pdf"
REPORT_TEXT_OUTPUT = TEXT_DIR / "trade-report_text.md"

for d in [DATA_DIR, CLEAN_DIR, RAW_DIR, REPORTS_DIR, IMG_DIR, TEXT_DIR, NOTEBOOKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Base folder   : {BASE_DIR}")
print(f"✅ Reports folder: {REPORTS_DIR}")
print(f"✅ Notebook cwd  : {Path.cwd().resolve()}")

# =============================================================================
# 3. REPORT TEXT LAYER
# =============================================================================
DEFAULT_REPORT_TEXT = {
    "executive_summary": """
[WRITE 1 SHORT EXECUTIVE PARAGRAPH HERE]

Suggested structure:
- What changed
- Why it matters
- What decision makers should do next
""".strip(),

    "market_context": """
[WRITE MARKET CONTEXT HERE]

Suggested ideas:
- importance of Canada-Brazil bilateral trade
- current economic context
- why this report matters now
""".strip(),

    "trade_overview": """
[WRITE INTERPRETATION FOR OVERVIEW HERE]

Suggested ideas:
- total trade growth or decline
- pace of change vs previous year
- whether momentum is broad-based or concentrated
""".strip(),

    "trade_balance": """
[WRITE INTERPRETATION FOR TRADE BALANCE HERE]

Suggested ideas:
- whether Canada is a net importer or exporter
- whether the imbalance is widening or narrowing
- strategic implications
""".strip(),

    "provincial_analysis": """
[WRITE INTERPRETATION FOR PROVINCIAL ANALYSIS HERE]

Suggested ideas:
- concentration of growth by province
- strongest and weakest regions
- dependency risk
""".strip(),

    "product_analysis": """
[WRITE INTERPRETATION FOR PRODUCT ANALYSIS HERE]

Suggested ideas:
- strongest product chapters
- declining sectors
- what this says about trade composition
""".strip(),

    "contribution_analysis": """
[WRITE INTERPRETATION FOR GROWTH CONTRIBUTION HERE]

Suggested ideas:
- few provinces driving most growth
- concentration and resilience considerations
""".strip(),

    "forecast": """
[WRITE INTERPRETATION FOR FORECAST HERE]

Suggested ideas:
- expected near-term direction
- confidence level
- caution on uncertainty
""".strip(),

    "opportunities": """
[WRITE INTERPRETATION FOR OPPORTUNITIES HERE]

Suggested ideas:
- lower-penetration provinces
- cross-provincial product expansion opportunities
- areas to prioritize
""".strip(),

    "recommendations": """
[WRITE 3 TO 5 RECOMMENDATIONS HERE]

Suggested format:
• Recommendation 1
• Recommendation 2
• Recommendation 3
• Recommendation 4
""".strip(),

    "technical_notes": """
[OPTIONAL TECHNICAL NOTES]

Suggested ideas:
- data source assumptions
- excluded chapters
- forecasting limitations
- nominal CAD values
""".strip()
}

def write_default_report_text(report_text_dict, output_path):
    lines = ["# Trade Report Writing Notes", ""]
    for key, value in report_text_dict.items():
        title = key.replace("_", " ").title()
        lines.append(f"## {title}")
        lines.append("")
        lines.append(value.strip())
        lines.append("")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines), encoding="utf-8")

def read_report_text(output_path, default_dict):
    if not output_path.exists():
        write_default_report_text(default_dict, output_path)
        print(f"✅ REPORT_TEXT created: {output_path}")

    raw = output_path.read_text(encoding="utf-8")
    parsed = {}
    current_key = None
    buffer = []

    section_map = {k.replace("_", " ").title(): k for k in default_dict.keys()}

    for line in raw.splitlines():
        if line.startswith("## "):
            if current_key is not None:
                parsed[current_key] = "\n".join(buffer).strip()
            title = line.replace("## ", "").strip()
            current_key = section_map.get(title)
            buffer = []
        else:
            if current_key is not None:
                buffer.append(line)

    if current_key is not None:
        parsed[current_key] = "\n".join(buffer).strip()

    for key, default_value in default_dict.items():
        if key not in parsed or not parsed[key].strip():
            parsed[key] = default_value

    return parsed

REPORT_TEXT = read_report_text(REPORT_TEXT_OUTPUT, DEFAULT_REPORT_TEXT)
print(f"✅ REPORT_TEXT loaded: {REPORT_TEXT_OUTPUT}")

# =============================================================================
# 4. HELPERS
# =============================================================================
def walk_find_file(base_paths, filename_contains=None, exact_names=None, suffix=None):
    base_paths = [Path(p) for p in base_paths if Path(p).exists()]
    exact_names_lower = [x.lower() for x in exact_names] if exact_names else None

    for base in base_paths:
        for root, _, files in os.walk(base):
            for f in files:
                f_low = f.lower()
                if exact_names_lower and f_low in exact_names_lower:
                    return Path(root) / f
                if filename_contains and filename_contains.lower() in f_low:
                    if suffix is None or f_low.endswith(suffix.lower()):
                        return Path(root) / f
    return None

def fmt_cad(v, d=1):
    if pd.isna(v):
        return "—"
    v = float(v)
    if abs(v) >= 1e9:
        return f"CA${v/1e9:.{d}f} BI"
    if abs(v) >= 1e6:
        return f"CA${v/1e6:.{d}f}M"
    if abs(v) >= 1e3:
        return f"CA${v/1e3:.{d}f}K"
    return f"CA${v:,.0f}"

def fmt_pct(v, d=1):
    if pd.isna(v):
        return "—"
    return f"{v:+.{d}f}%"

def safe_div(a, b):
    if b is None or b == 0 or pd.isna(b):
        return np.nan
    return a / b

def safe_pct_change(new, old):
    if old is None or old == 0 or pd.isna(old):
        return np.nan
    return ((new / old) - 1) * 100

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=190, bbox_inches="tight", facecolor="white")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def save_fig(fig, name):
    path = IMG_DIR / f"{name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=190, bbox_inches="tight", facecolor="white")
    return path

def df_to_html_table(df_, index=False):
    if df_ is None or len(df_) == 0:
        return "<p>No data available.</p>"
    return df_.to_html(index=index, classes="data-table", border=0, escape=False)

def markdown_like_to_html(text):
    if not isinstance(text, str):
        return ""
    lines = text.strip().splitlines()
    html_parts = []
    in_list = False

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("• ") or stripped.startswith("- "):
            if not in_list:
                html_parts.append("<ul>")
                in_list = True
            html_parts.append(f"<li>{stripped[2:].strip()}</li>")
        elif stripped == "":
            if in_list:
                html_parts.append("</ul>")
                in_list = False
            html_parts.append("<br>")
        else:
            if in_list:
                html_parts.append("</ul>")
                in_list = False
            html_parts.append(f"<p>{stripped}</p>")

    if in_list:
        html_parts.append("</ul>")

    return "\n".join(html_parts)

def clean_axes(ax, keep_left_spine=False, keep_y=False):
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    if keep_left_spine:
        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_color(C_DKGRAY)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_color(C_DKGRAY)
    if not keep_y:
        ax.yaxis.set_visible(False)
    ax.xaxis.set_ticks_position("none")
    ax.tick_params(axis="x", labelsize=9, colors=C_DKGRAY)
    ax.set_facecolor(C_WHITE)

def safe_label(text, max_len=36):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 1] + "…"

def add_text_subtitle(fig, parts, anchor=(0.5, 0.92), fontsize=11):
    children = [
        TextArea(
            txt,
            textprops=dict(color=col, fontsize=fontsize, fontweight=weight, fontstyle="italic")
        )
        for txt, col, weight in parts
    ]
    box = HPacker(children=children, align="baseline", pad=0, sep=2)
    fig.add_artist(
        AnchoredOffsetbox(
            loc="upper center",
            child=box,
            pad=0,
            frameon=False,
            bbox_to_anchor=anchor,
            bbox_transform=fig.transFigure,
            borderpad=0
        )
    )

# =============================================================================
# 5. DATA LOAD
# =============================================================================
dataset_path = walk_find_file(
    [BASE_DIR, DATA_DIR, CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
    exact_names=["dataset_clean.csv", "clean_dataset.csv", "final_dataset.csv"]
)

if dataset_path is None:
    dataset_path = walk_find_file(
        [BASE_DIR, DATA_DIR, CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
        filename_contains="clean",
        suffix=".csv"
    )

if dataset_path is None:
    raise FileNotFoundError(
        "dataset_clean.csv not found.\n"
        "Expected example path:\n"
        "/content/drive/MyDrive/canada-brazil-trade-report/data/clean/dataset_clean.csv"
    )

hs_map_path = walk_find_file(
    [BASE_DIR, DATA_DIR, CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
    exact_names=["hs_mapping.xlsx"]
)

print(f"✅ Loading dataset: {dataset_path}")
if hs_map_path:
    print(f"✅ Loading HS map : {hs_map_path}")
else:
    print("⚠️ hs_mapping.xlsx not found — using fallback chapter labels.")

df = pd.read_csv(dataset_path)

# =============================================================================
# 6. STANDARDIZE COLUMNS
# =============================================================================
required_cols = ["flow", "value"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in dataset: {missing_required}")

if "period_m" in df.columns:
    df["Period"] = pd.to_datetime(df["period_m"], errors="coerce")
elif "period" in df.columns:
    df["Period"] = pd.to_datetime(df["period"], errors="coerce")
elif "date" in df.columns:
    df["Period"] = pd.to_datetime(df["date"], errors="coerce")
else:
    raise ValueError("Missing a usable date column: period_m / period / date")

df = df.dropna(subset=["Period"]).copy()

if "year" in df.columns:
    df["Year"] = pd.to_numeric(df["year"], errors="coerce").fillna(df["Period"].dt.year).astype(int)
else:
    df["Year"] = df["Period"].dt.year.astype(int)

if "province" in df.columns:
    df["Province"] = df["province"].astype(str).str.strip()
else:
    df["Province"] = "Unknown"

df["Month"] = df["Period"].dt.to_period("M")
df["Month_Num"] = df["Period"].dt.month
df["Value"] = pd.to_numeric(df["value"], errors="coerce").fillna(0)
df["flow"] = df["flow"].astype(str).str.strip().str.lower()

if "hs8" in df.columns:
    hs8_num = pd.to_numeric(df["hs8"], errors="coerce")
    df = df[~hs8_num.isin([98, 99])].copy()

if "hs2" in df.columns:
    hs2_num = pd.to_numeric(df["hs2"], errors="coerce")
    df = df[(hs2_num < 97) | (hs2_num.isna())].copy()

if "is_ch98" in df.columns:
    df = df[df["is_ch98"] != True].copy()

df["flow"] = df["flow"].replace({
    "export": "export",
    "exports": "export",
    "import": "import",
    "imports": "import"
})

if hs_map_path is not None and "hs8" in df.columns:
    try:
        df_hs = pd.read_excel(hs_map_path)
        df_hs.columns = [str(c).strip() for c in df_hs.columns]

        if len(df_hs.columns) >= 4:
            df_hs = df_hs.iloc[:, :4].copy()
            df_hs.columns = ["Section", "Section_Name", "Chapter", "Chapter_Name"]
            df_hs["Chapter"] = (
                df_hs["Chapter"]
                .astype(str)
                .str.extract(r"(\d+)")[0]
                .str.zfill(2)
            )
            hs_lkp = df_hs.drop_duplicates("Chapter").set_index("Chapter")["Chapter_Name"].to_dict()

            df["hs_chapter"] = (
                pd.to_numeric(df["hs8"], errors="coerce")
                .fillna(0)
                .astype(int)
                .astype(str)
                .str.zfill(2)
            )
            df["Chapter_Name"] = df["hs_chapter"].map(hs_lkp).fillna("Other")
        else:
            raise ValueError("HS mapping format not recognized.")
    except Exception as e:
        print(f"⚠️ HS mapping fallback used: {e}")
        if "chapter_name" in df.columns:
            df["Chapter_Name"] = df["chapter_name"].astype(str).str.strip().replace("", "Unknown")
        elif "description" in df.columns:
            df["Chapter_Name"] = df["description"].astype(str).str.strip().replace("", "Unknown")
        else:
            df["Chapter_Name"] = "Unknown"
else:
    if "chapter_name" in df.columns:
        df["Chapter_Name"] = df["chapter_name"].astype(str).str.strip().replace("", "Unknown")
    elif "description" in df.columns:
        df["Chapter_Name"] = df["description"].astype(str).str.strip().replace("", "Unknown")
    else:
        df["Chapter_Name"] = "Unknown"

# =============================================================================
# 7. STYLE SYSTEM
# =============================================================================
C_DKGREEN  = "#004D25"
C_LTGREEN  = "#99CC33"
C_RED      = "#E62310"
C_YELLOW   = "#FFCC22"
C_GRAY     = "#CCCCCC"
C_DKGRAY   = "#555555"
C_WHITE    = "#FFFFFF"
C_ROWALT   = "#F5F5F5"
C_BG       = "#F7FAF7"

plt.rcParams["figure.facecolor"] = C_WHITE
plt.rcParams["axes.facecolor"] = C_WHITE
plt.rcParams["savefig.facecolor"] = C_WHITE
plt.rcParams["font.family"] = "DejaVu Sans"

# =============================================================================
# 8. CORE AGGREGATIONS
# =============================================================================
available_years = sorted(df["Year"].dropna().unique().tolist())
if len(available_years) < 2:
    raise ValueError("At least two years are required to compare performance.")

YEAR1 = available_years[-2]
YEAR2 = available_years[-1]

print(f"📅 Comparing {YEAR1} vs {YEAR2}")

ann = df.groupby(["Year", "flow"])["Value"].sum().unstack(fill_value=0)
for col in ["export", "import"]:
    if col not in ann.columns:
        ann[col] = 0

ann["total"] = ann["export"] + ann["import"]
ann["balance"] = ann["export"] - ann["import"]

total_2024 = ann.loc[YEAR1, "total"]
total_2025 = ann.loc[YEAR2, "total"]
exp_2024 = ann.loc[YEAR1, "export"]
exp_2025 = ann.loc[YEAR2, "export"]
imp_2024 = ann.loc[YEAR1, "import"]
imp_2025 = ann.loc[YEAR2, "import"]
growth_pct = safe_pct_change(total_2025, total_2024)

monthly = (
    df.groupby("Month")["Value"]
    .sum()
    .reset_index()
    .rename(columns={"Value": "total"})
)
monthly["Period"] = monthly["Month"].dt.to_timestamp()
monthly["Year"] = monthly["Period"].dt.year
monthly = monthly.sort_values("Period")

monthly_flow = (
    df.groupby(["Period", "flow"])["Value"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"export": "exports", "import": "imports"})
)
if "exports" not in monthly_flow.columns:
    monthly_flow["exports"] = 0
if "imports" not in monthly_flow.columns:
    monthly_flow["imports"] = 0

monthly_flow["balance"] = monthly_flow["exports"] - monthly_flow["imports"]
monthly_flow = monthly_flow.sort_values("Period")

prov = (
    df.groupby(["Year", "Province"])["Value"]
    .sum()
    .reset_index()
    .pivot(index="Province", columns="Year", values="Value")
    .fillna(0)
)

if YEAR1 not in prov.columns:
    prov[YEAR1] = 0
if YEAR2 not in prov.columns:
    prov[YEAR2] = 0

prov = prov.rename(columns={YEAR1: "v2024", YEAR2: "v2025"})
prov["growth_pct"] = (prov["v2025"] - prov["v2024"]) / prov["v2024"].replace(0, np.nan) * 100
prov["change_abs"] = prov["v2025"] - prov["v2024"]
prov["contribution"] = np.where(
    prov["change_abs"].sum() != 0,
    prov["change_abs"] / prov["change_abs"].sum() * 100,
    0
)
prov["share_2024"] = np.where(prov["v2024"].sum() != 0, prov["v2024"] / prov["v2024"].sum() * 100, 0)
prov["share_2025"] = np.where(prov["v2025"].sum() != 0, prov["v2025"] / prov["v2025"].sum() * 100, 0)
prov = prov.reset_index()

chap = (
    df.groupby(["Year", "Chapter_Name"])["Value"]
    .sum()
    .reset_index()
    .pivot(index="Chapter_Name", columns="Year", values="Value")
    .fillna(0)
)

if YEAR1 not in chap.columns:
    chap[YEAR1] = 0
if YEAR2 not in chap.columns:
    chap[YEAR2] = 0

chap = chap.rename(columns={YEAR1: "v2024", YEAR2: "v2025"})
chap["growth_pct"] = (chap["v2025"] - chap["v2024"]) / chap["v2024"].replace(0, np.nan) * 100
chap["change_abs"] = chap["v2025"] - chap["v2024"]
chap["share_2025"] = np.where(chap["v2025"].sum() != 0, chap["v2025"] / chap["v2025"].sum() * 100, 0)
chap = chap.reset_index()

MIN_BASELINE_THRESHOLD = 5_000_000
chap_filt = chap[(chap["v2024"] >= MIN_BASELINE_THRESHOLD) | (chap["v2025"] >= MIN_BASELINE_THRESHOLD)].copy()
chap_filt = chap_filt.sort_values("v2025", ascending=False)

# =============================================================================
# 9. SUPPORTING TABLES
# =============================================================================
trade_stats_df = pd.DataFrame({
    "Metric": [
        f"Total Trade {YEAR1}",
        f"Total Trade {YEAR2}",
        "Total Trade Growth",
        f"Exports {YEAR1}",
        f"Exports {YEAR2}",
        "Exports Growth",
        f"Imports {YEAR1}",
        f"Imports {YEAR2}",
        "Imports Growth",
        f"Trade Balance {YEAR2}",
    ],
    "Value": [
        fmt_cad(total_2024),
        fmt_cad(total_2025),
        fmt_pct(growth_pct),
        fmt_cad(exp_2024),
        fmt_cad(exp_2025),
        fmt_pct(safe_pct_change(exp_2025, exp_2024)),
        fmt_cad(imp_2024),
        fmt_cad(imp_2025),
        fmt_pct(safe_pct_change(imp_2025, imp_2024)),
        fmt_cad(exp_2025 - imp_2025),
    ]
})

prov_table = (
    prov[["Province", "v2024", "v2025", "growth_pct", "change_abs", "share_2025", "contribution"]]
    .sort_values("v2025", ascending=False)
    .copy()
)
prov_table.columns = [
    "Province",
    f"{YEAR1} Value",
    f"{YEAR2} Value",
    "Growth %",
    "Abs Change",
    f"{YEAR2} Share %",
    "Contribution %"
]
for c in [f"{YEAR1} Value", f"{YEAR2} Value", "Abs Change"]:
    prov_table[c] = prov_table[c].apply(fmt_cad)
for c in ["Growth %", f"{YEAR2} Share %", "Contribution %"]:
    prov_table[c] = prov_table[c].apply(fmt_pct)

chapter_table = (
    chap_filt[["Chapter_Name", "v2024", "v2025", "growth_pct", "change_abs", "share_2025"]]
    .sort_values("v2025", ascending=False)
    .head(15)
    .copy()
)
chapter_table.columns = [
    "Chapter",
    f"{YEAR1} Value",
    f"{YEAR2} Value",
    "Growth %",
    "Abs Change",
    f"{YEAR2} Share %"
]
for c in [f"{YEAR1} Value", f"{YEAR2} Value", "Abs Change"]:
    chapter_table[c] = chapter_table[c].apply(fmt_cad)
for c in ["Growth %", f"{YEAR2} Share %"]:
    chapter_table[c] = chapter_table[c].apply(fmt_pct)

concentration_df = (
    prov.sort_values("v2025", ascending=False)[["Province", "v2025", "share_2025", "growth_pct", "change_abs"]]
    .copy()
)
concentration_df.columns = ["Province", f"{YEAR2} Value", f"{YEAR2} Share %", "Growth %", "Abs Change"]
for c in [f"{YEAR2} Value", "Abs Change"]:
    concentration_df[c] = concentration_df[c].apply(fmt_cad)
for c in [f"{YEAR2} Share %", "Growth %"]:
    concentration_df[c] = concentration_df[c].apply(fmt_pct)

# =============================================================================
# 10. FORECAST + OPPORTUNITY TABLES
# =============================================================================
monthly_total = monthly[monthly["Year"].isin([YEAR1, YEAR2])].copy()
monthly_total["Month_Num"] = monthly_total["Period"].dt.month

seasonality = (
    monthly_total[monthly_total["Year"] == YEAR2]
    .groupby("Month_Num")["total"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)
if seasonality.sum() == 0:
    seasonality = pd.Series([1] * 12, index=range(1, 13))
seasonality_share = seasonality / seasonality.sum()

forecast_growth = safe_pct_change(total_2025, total_2024)
forecast_total_2026 = total_2025 * (1 + (0 if pd.isna(forecast_growth) else forecast_growth / 100))
forecast_monthly = pd.DataFrame({
    "Month_Num": range(1, 13),
    "Forecast_Value": (seasonality_share.values * forecast_total_2026)
})
forecast_monthly["Month"] = pd.to_datetime(
    [f"{YEAR2 + 1}-{m:02d}-01" for m in forecast_monthly["Month_Num"]]
)
forecast_monthly["Month_Label"] = forecast_monthly["Month"].dt.strftime("%b %Y")

forecast_table = forecast_monthly[["Month_Label", "Forecast_Value"]].copy()
forecast_table.columns = ["Month", f"Forecast {YEAR2 + 1}"]
forecast_table[f"Forecast {YEAR2 + 1}"] = forecast_table[f"Forecast {YEAR2 + 1}"].apply(fmt_cad)

# Strategic opportunities
prov_rank = prov[["Province", "v2025", "growth_pct", "share_2025"]].copy()
prov_rank["Province_Strength"] = np.where(
    (prov_rank["growth_pct"].fillna(0) > 0) & (prov_rank["share_2025"] > prov_rank["share_2025"].median()),
    "Core Growth Market",
    np.where(
        (prov_rank["growth_pct"].fillna(0) > 0),
        "Emerging Growth Market",
        "Underperforming / Watch"
    )
)

top_chapters = chap_filt.sort_values("v2025", ascending=False).head(8)[["Chapter_Name", "v2025", "growth_pct"]].copy()
top_chapters["Chapter_Name"] = top_chapters["Chapter_Name"].apply(safe_label)

expansion_rows = []
for _, p_row in prov_rank.sort_values(["growth_pct", "share_2025"], ascending=[False, False]).head(6).iterrows():
    for _, c_row in top_chapters.head(3).iterrows():
        expansion_rows.append({
            "Province": p_row["Province"],
            "Province Status": p_row["Province_Strength"],
            "Suggested Chapter": c_row["Chapter_Name"],
            "Chapter Growth %": fmt_pct(c_row["growth_pct"]),
            "Rationale": (
                "Positive regional momentum with exposure potential to higher-value trade categories."
                if p_row["growth_pct"] > 0 else
                "Use selectively where existing demand base supports diversification."
            )
        })

expansion_df = pd.DataFrame(expansion_rows).drop_duplicates(subset=["Province", "Suggested Chapter"]).head(12)

# =============================================================================
# 11. EXECUTIVE SUMMARY AUTO TEXT
# =============================================================================
top_prov_for_summary = prov.sort_values("change_abs", ascending=False).iloc[0] if len(prov) else None
top_chap_for_summary = chap_filt.sort_values("change_abs", ascending=False).iloc[0] if len(chap_filt) else None
balance_2025 = exp_2025 - imp_2025

executive_summary_auto = (
    f"Canada-Brazil merchandise trade reached {fmt_cad(total_2025)} in {YEAR2}, "
    f"{fmt_pct(growth_pct)} versus {YEAR1}. Exports totaled {fmt_cad(exp_2025)} and "
    f"imports reached {fmt_cad(imp_2025)}, leaving a trade balance of {fmt_cad(balance_2025)}. "
    f"{'' if top_prov_for_summary is None else f'The largest provincial contribution came from {top_prov_for_summary['Province']}, '}"

    f"{'' if top_chap_for_summary is None else f'while {top_chap_for_summary['Chapter_Name']} was the strongest product chapter by absolute change. '}"
    f"The baseline outlook for {YEAR2 + 1} implies total trade near {fmt_cad(forecast_total_2026)} if current momentum and seasonal distribution persist."
)

# =============================================================================
# 12. CHARTS
# =============================================================================
def chart_q1_annual_trade():
    exp_24_val = exp_2024 / 1e9
    exp_25_val = exp_2025 / 1e9
    imp_24_val = imp_2024 / 1e9
    imp_25_val = imp_2025 / 1e9

    growth_exp = safe_pct_change(exp_2025, exp_2024)
    growth_imp = safe_pct_change(imp_2025, imp_2024)

    fig, ax = plt.subplots(figsize=(12, 7.4))
    fig.subplots_adjust(top=0.88, bottom=0.20, left=0.10, right=0.90)

    x = np.array([0, 2.7])
    w = 0.85
    gap = 0.15

    bars_24 = ax.bar(x - w/2 - gap, [exp_24_val, imp_24_val], width=w, color=C_LTGREEN, zorder=3)
    bars_25 = ax.bar(x + w/2 + gap, [exp_25_val, imp_25_val], width=w, color=C_DKGREEN, zorder=3)

    for bar in list(bars_24) + list(bars_25):
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2, h - 0.35,
            f"CA${h:.1f} BI",
            ha="center", va="top",
            fontsize=13, color=C_WHITE, fontweight="bold"
        )

    def add_growth_annotation(x_pos, val_24, val_25, growth_pct_local):
        if pd.isna(growth_pct_local):
            return
        y_bridge = max(val_24, val_25) + 1.0
        x_start = x_pos - w/2 - gap
        x_end = x_pos + w/2 + gap
        x_mid = (x_start + x_end) / 2
        ax.plot([x_start, x_start, x_mid - 0.22], [val_24 + 0.15, y_bridge, y_bridge], color=C_DKGRAY, lw=1)
        ax.annotate(
            "", xy=(x_end, y_bridge), xytext=(x_mid + 0.22, y_bridge),
            arrowprops=dict(arrowstyle="->", color=C_DKGRAY, lw=1, shrinkA=0, shrinkB=0)
        )
        ax.text(
            x_mid, y_bridge,
            f"{growth_pct_local:+.1f}%",
            ha="center", va="center",
            color=C_DKGREEN if growth_pct_local >= 0 else C_RED,
            fontweight="bold", fontsize=17, backgroundcolor="white"
        )

    add_growth_annotation(x[0], exp_24_val, exp_25_val, growth_exp)
    add_growth_annotation(x[1], imp_24_val, imp_25_val, growth_imp)

    ax.plot([x[0] - 1.05, x[0] + 1.05], [0, 0], color=C_GRAY, lw=1.5, zorder=2)
    ax.plot([x[1] - 1.05, x[1] + 1.05], [0, 0], color=C_GRAY, lw=1.5, zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels([])

    ax.text(x[0], -max(exp_25_val, imp_25_val)*0.15, "EXPORTS", fontsize=15, fontweight="bold", color=C_DKGRAY, ha="center")
    ax.text(x[1], -max(exp_25_val, imp_25_val)*0.15, "IMPORTS", fontsize=15, fontweight="bold", color=C_DKGRAY, ha="center")

    for i in range(len(x)):
        ax.text(x[i] - w/2 - gap, -0.45, str(YEAR1), ha="center", va="top", fontsize=12, color=C_LTGREEN, fontweight="bold")
        ax.text(x[i] + w/2 + gap, -0.45, str(YEAR2), ha="center", va="top", fontsize=12, color=C_DKGREEN, fontweight="bold")

    ax.set_ylim(0, max(exp_25_val, imp_25_val, exp_24_val, imp_24_val) + 2.2)
    ax.set_xlim(x[0] - 1.0, x[1] + 1.0)
    clean_axes(ax)
    ax.spines["bottom"].set_visible(False)

    total_25_bn = total_2025 / 1e9
    fig.text(
        0.50, 0.96,
        f"Canada’s Exports and Imports with Brazil reached record levels in {YEAR2}",
        fontsize=20, fontweight="bold", color=C_DKGREEN, ha="center"
    )

    subtitle_parts = [
        ("Total trade value grew to ", C_DKGRAY, "normal"),
        (f"CA${total_25_bn:.1f} BI", C_DKGREEN, "bold"),
        (", driven by a ", C_DKGRAY, "normal"),
        (fmt_pct(growth_exp), C_DKGREEN if (pd.notna(growth_exp) and growth_exp >= 0) else C_RED, "bold"),
        (" increase in exports and ", C_DKGRAY, "normal"),
        (fmt_pct(growth_imp), C_DKGREEN if (pd.notna(growth_imp) and growth_imp >= 0) else C_RED, "bold"),
        (" in imports.", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, subtitle_parts, anchor=(0.5, 0.92), fontsize=12)
    return fig

def chart_q1b_monthly_yoy():
    df_plot = df.copy()
    df_plot["Month_Num"] = df_plot["Period"].dt.month
    trend_data = df_plot.groupby(["Year", "Month_Num"])["Value"].sum().unstack(level=0)

    v24 = (trend_data.get(YEAR1, pd.Series(dtype=float)) / 1e9).reindex(range(1, 13), fill_value=0)
    v25 = (trend_data.get(YEAR2, pd.Series(dtype=float)) / 1e9).reindex(range(1, 13), fill_value=0)
    yoy_diff = ((v25 / v24.replace(0, np.nan)) - 1).fillna(0) * 100

    months_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    values = yoy_diff.values

    fig, ax = plt.subplots(figsize=(13, 6.4))
    fig.subplots_adjust(top=0.84, bottom=0.17, left=0.08, right=0.96)

    bars = ax.bar(
        range(len(values)),
        values,
        color=[C_DKGREEN if v >= 0 else C_RED for v in values],
        width=0.68
    )

    for bar in bars:
        height = bar.get_height()
        label_color = C_DKGREEN if height >= 0 else C_RED
        va_type = "bottom" if height >= 0 else "top"
        offset = 1.0 if height >= 0 else -1.0
        ax.text(
            bar.get_x() + bar.get_width()/2, height + offset,
            f"{height:+.1f}%", ha="center", va=va_type,
            fontsize=9.5, fontweight="bold", color=label_color
        )

    ax.axhline(0, color="black", linewidth=1.1)
    ax.set_ylim(min(values) - 15, max(values) + 22 if len(values) else 10)
    ax.set_xticks(range(len(months_labels)))
    ax.set_xticklabels(months_labels, fontsize=11, color=C_DKGRAY, fontweight="bold")
    ax.set_ylabel("Year-over-Year Growth (%)", fontsize=10.5, color=C_DKGRAY, fontweight="bold")
    ax.grid(axis="y", ls="--", alpha=0.30, color=C_GRAY)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)

    fig.text(
        0.5, 0.96,
        f"Monthly Performance: {YEAR2} vs {YEAR1} Comparative Analysis",
        ha="center", fontsize=19, color=C_DKGREEN, fontweight="bold"
    )
    sub_parts = [
        ("Despite occasional market shifts, monthly trade volumes stayed above ", C_DKGRAY, "normal"),
        (str(YEAR1), C_DKGREEN, "bold"),
        (" levels in most periods.", C_DKGRAY, "normal")
    ]
    add_text_subtitle(fig, sub_parts, anchor=(0.5, 0.90), fontsize=11)
    return fig

def chart_q2_province_growth_decline():
    prov_sorted = prov.sort_values("growth_pct", ascending=True).copy()

    if len(prov_sorted) == 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No province data available.", ha="center", va="center")
        ax.axis("off")
        return fig

    top_prov = prov_sorted.loc[prov_sorted["growth_pct"].idxmax()]
    top_prov_name = top_prov["Province"]
    top_prov_val = top_prov["v2025"] / 1e9
    top_prov_pct = top_prov["growth_pct"]

    fig, ax = plt.subplots(figsize=(13, max(6, len(prov_sorted) * 0.55)))
    fig.subplots_adjust(top=0.82, bottom=0.10, left=0.20, right=0.82)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    y_pos = np.arange(len(prov_sorted))
    values = prov_sorted["growth_pct"].fillna(0).values
    colors = [C_DKGREEN if v >= 0 else C_RED for v in values]

    bars = ax.barh(y_pos, values, color=colors, height=0.65, edgecolor=C_WHITE)

    x_min = min(values) - 15
    x_max = max(values) + 15
    ax.set_xlim(x_min, x_max)

    for bar, v in zip(bars, values):
        x_offset = 1.5 if v >= 0 else -1.5
        ha = "left" if v >= 0 else "right"
        ax.text(
            v + x_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{v:+.1f}%",
            va="center",
            ha=ha,
            fontsize=9.5,
            fontweight="bold",
            color=C_DKGREEN if v >= 0 else C_RED,
        )

    ax.axvline(0, color=C_DKGRAY, linewidth=1.2, zorder=3)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(prov_sorted["Province"].values, fontsize=10, color=C_DKGRAY)
    ax.set_xlabel("Year-over-Year Growth (%)", fontsize=10, color=C_DKGRAY, labelpad=8)

    x_callout = x_max + 4.0
    for i, (_, row) in enumerate(prov_sorted.iterrows()):
        ax.text(
            x_callout,
            i,
            f"CA${row['v2025']/1e9:.2f} BI",
            va="center",
            ha="left",
            fontsize=8.5,
            color=C_DKGRAY,
        )

    ax.text(
        x_callout,
        len(prov_sorted) - 0.15,
        "2025 Volume",
        va="bottom",
        ha="left",
        fontsize=9,
        color=C_DKGRAY,
        fontstyle="italic",
    )

    ax.set_xlim(x_min, x_callout + 5.5)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(C_DKGRAY)
    ax.tick_params(axis="x", colors=C_DKGRAY, labelsize=9)
    ax.tick_params(axis="y", left=False)
    ax.grid(axis="x", ls="--", alpha=0.25, color=C_GRAY)

    fig.text(
        0.50,
        0.96,
        f"{top_prov_name} Leads Provincial Growth at {top_prov_pct:+.1f}% — CA${top_prov_val:.2f}B in {YEAR2}",
        ha="center",
        fontsize=18,
        fontweight="bold",
        color=C_DKGREEN,
    )

    n_grow = int((prov_sorted["growth_pct"] > 0).sum())
    n_shrink = int((prov_sorted["growth_pct"] < 0).sum())
    sub_parts_q2 = [
        (f"{n_grow} provinces expanded", C_DKGREEN, "bold"),
        (" trade flows while ", C_DKGRAY, "normal"),
        (f"{n_shrink} contracted", C_RED, "bold"),
        (", signalling diverging regional investment priorities.", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, sub_parts_q2, anchor=(0.50, 0.91), fontsize=11)
    return fig

def chart_q3_trade_balance():
    mf = monthly_flow.copy()
    mf["Year"] = pd.to_datetime(mf["Period"]).dt.year
    mf["month_num"] = pd.to_datetime(mf["Period"]).dt.month
    mf = mf[mf["Year"].isin([YEAR1, YEAR2])].sort_values("Period").copy()
    mf["label"] = pd.to_datetime(mf["Period"]).dt.strftime("%b %Y")

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1,
        figsize=(14, 9),
        gridspec_kw={"height_ratios": [3, 1.5]},
        sharex=True
    )
    fig.subplots_adjust(top=0.86, bottom=0.14, left=0.08, right=0.96, hspace=0.08)
    fig.patch.set_facecolor(C_WHITE)

    x_vals = np.arange(len(mf))

    ax_top.plot(
        x_vals, mf["exports"] / 1e9,
        color=C_LTGREEN, lw=2.5, marker="o", ms=4, label="Exports", zorder=3
    )
    ax_top.plot(
        x_vals, mf["imports"] / 1e9,
        color=C_DKGREEN, lw=2.5, marker="s", ms=4, label="Imports", zorder=3
    )

    ax_top.fill_between(
        x_vals,
        mf["exports"].values / 1e9,
        mf["imports"].values / 1e9,
        where=(mf["imports"].values >= mf["exports"].values),
        alpha=0.12,
        color=C_RED,
    )
    ax_top.fill_between(
        x_vals,
        mf["exports"].values / 1e9,
        mf["imports"].values / 1e9,
        where=(mf["exports"].values > mf["imports"].values),
        alpha=0.12,
        color=C_LTGREEN,
    )

    year2_start = mf[mf["Year"] == YEAR2].index
    if len(year2_start) > 0:
        divider_x = mf.index.get_loc(year2_start[0])
        ax_top.axvline(divider_x - 0.5, color=C_DKGRAY, lw=1, ls="--", alpha=0.5)
        ylim_top = max((mf["exports"].max() / 1e9), (mf["imports"].max() / 1e9))
        ax_top.text(
            divider_x - 0.45, ylim_top * 0.96,
            f"  {YEAR2} →",
            fontsize=9, color=C_DKGRAY, fontstyle="italic"
        )

    ax_top.legend(loc="upper left", frameon=False, fontsize=9, ncol=2)
    ax_top.set_ylabel("CA$ Billion", fontsize=10, color=C_DKGRAY, labelpad=6)
    ax_top.tick_params(axis="y", colors=C_DKGRAY, labelsize=9)
    ax_top.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"CA${v:.1f}B"))
    ax_top.grid(axis="y", ls="--", alpha=0.20, color=C_GRAY)

    for spine in ["top", "right"]:
        ax_top.spines[spine].set_visible(False)
    ax_top.spines["bottom"].set_visible(False)
    ax_top.spines["left"].set_color(C_GRAY)
    ax_top.set_facecolor(C_WHITE)
    ax_top.tick_params(bottom=False, labelbottom=False)

    balance_vals = mf["balance"].values / 1e9
    b_colors = [C_DKGREEN if v >= 0 else C_RED for v in balance_vals]
    ax_bot.bar(x_vals, balance_vals, color=b_colors, width=0.75, zorder=3)
    ax_bot.axhline(0, color=C_DKGRAY, lw=1.2)
    ax_bot.set_ylabel("Net Balance\n(CA$ B)", fontsize=9, color=C_DKGRAY, labelpad=6)
    ax_bot.tick_params(axis="y", colors=C_DKGRAY, labelsize=8)
    ax_bot.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.1f}"))

    for spine in ["top", "right"]:
        ax_bot.spines[spine].set_visible(False)
    ax_bot.spines["left"].set_color(C_GRAY)
    ax_bot.spines["bottom"].set_color(C_DKGRAY)
    ax_bot.set_facecolor(C_WHITE)
    ax_bot.grid(axis="y", ls="--", alpha=0.20, color=C_GRAY)

    tick_step = 3
    tick_idx = [i for i in range(len(mf)) if i % tick_step == 0]
    ax_bot.set_xticks(tick_idx)
    ax_bot.set_xticklabels(
        [mf["label"].iloc[i] for i in tick_idx],
        fontsize=8.5, color=C_DKGRAY, rotation=30, ha="right"
    )

    latest_bal = mf["balance"].iloc[-1] / 1e9
    latest_lbl = mf["label"].iloc[-1]
    pos_months = int((mf["balance"] > 0).sum())
    neg_months = int((mf["balance"] < 0).sum())

    headline_color = C_RED if latest_bal < 0 else C_DKGREEN
    fig.text(
        0.50, 0.95,
        f"Canada runs a persistent trade deficit with Brazil — {latest_lbl} balance: CA${latest_bal:+.2f} BI",
        ha="center", fontsize=16, fontweight="bold", color=headline_color
    )

    sub_parts_q3 = [
        (f"{neg_months} months", C_RED, "bold"),
        (" showed an import surplus vs. ", C_DKGRAY, "normal"),
        (f"{pos_months} months", C_DKGREEN, "bold"),
        (" with an export surplus — a structural imbalance requiring product diversification.", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, sub_parts_q3, anchor=(0.50, 0.90), fontsize=11)
    return fig

def chart_q4_chapter_quadrant():
    q4 = chap_filt[["Chapter_Name", "v2024", "v2025", "growth_pct", "change_abs"]].copy()
    q4 = q4.dropna(subset=["growth_pct"]).copy()

    if len(q4) == 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No chapter data available.", ha="center", va="center")
        ax.axis("off")
        return fig

    growth_cap = 200
    q4["growth_plot"] = q4["growth_pct"].clip(-growth_cap, growth_cap)
    size_scale = np.where(q4["v2025"] > 0, (q4["v2025"] / q4["v2025"].max()) * 1800 + 90, 90)

    x_mid = q4["v2025"].median()
    y_mid = 0

    fig, ax = plt.subplots(figsize=(13, 8))
    fig.subplots_adjust(top=0.84, bottom=0.12, left=0.10, right=0.96)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    colors = []
    for _, row in q4.iterrows():
        if row["v2025"] >= x_mid and row["growth_plot"] >= y_mid:
            colors.append(C_DKGREEN)
        elif row["v2025"] >= x_mid and row["growth_plot"] < y_mid:
            colors.append(C_RED)
        elif row["v2025"] < x_mid and row["growth_plot"] >= y_mid:
            colors.append(C_LTGREEN)
        else:
            colors.append(C_GRAY)

    ax.scatter(
        q4["v2025"] / 1e9,
        q4["growth_plot"],
        s=size_scale,
        c=colors,
        alpha=0.75,
        edgecolors="white",
        linewidths=0.8
    )

    q4_sorted = q4.sort_values(["v2025", "change_abs"], ascending=False).head(10)
    for _, row in q4_sorted.iterrows():
        ax.text(
            row["v2025"] / 1e9,
            row["growth_plot"] + 3,
            safe_label(row["Chapter_Name"], 28),
            fontsize=8.5,
            color=C_DKGRAY,
            ha="center"
        )

    ax.axvline(x_mid / 1e9, color=C_DKGRAY, lw=1.1, ls="--", alpha=0.5)
    ax.axhline(y_mid, color=C_DKGRAY, lw=1.1, ls="--", alpha=0.5)

    ax.set_xlabel(f"{YEAR2} Trade Value (CA$ Billion)", fontsize=10, color=C_DKGRAY)
    ax.set_ylabel("YoY Growth (%)", fontsize=10, color=C_DKGRAY)
    ax.grid(ls="--", alpha=0.22, color=C_GRAY)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(C_GRAY)
    ax.spines["bottom"].set_color(C_GRAY)
    ax.tick_params(colors=C_DKGRAY)

    fig.text(
        0.50, 0.96,
        "HS Chapter Winners vs. Risks — Large categories are separating into growth leaders and declining exposures",
        ha="center", fontsize=17, fontweight="bold", color=C_DKGREEN
    )
    sub_parts = [
        ("Upper-right = high volume / high growth", C_DKGREEN, "bold"),
        (" | lower-right = large but declining trade categories", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, sub_parts, anchor=(0.50, 0.90), fontsize=11)
    return fig

def chart_q5_concentration():
    q5 = prov.sort_values("v2025", ascending=False).copy()
    q5 = q5.head(10).copy()

    fig, ax = plt.subplots(figsize=(13, 7))
    fig.subplots_adjust(top=0.84, bottom=0.17, left=0.08, right=0.96)

    x = np.arange(len(q5))
    bars = ax.bar(x, q5["share_2025"], color=C_DKGREEN, width=0.68)

    cum_share = q5["share_2025"].cumsum()
    ax2 = ax.twinx()
    ax2.plot(x, cum_share, color=C_RED, lw=2.5, marker="o", ms=4)
    ax2.set_ylim(0, 105)
    ax2.tick_params(axis="y", colors=C_RED, labelsize=9)
    ax2.set_ylabel("Cumulative Share (%)", color=C_RED, fontsize=10)

    for i, bar in enumerate(bars):
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            h + 0.8,
            f"{h:.1f}%",
            ha="center", va="bottom",
            fontsize=9, fontweight="bold", color=C_DKGREEN
        )

    ax.set_xticks(x)
    ax.set_xticklabels(q5["Province"], rotation=30, ha="right", fontsize=9, color=C_DKGRAY)
    ax.set_ylabel(f"{YEAR2} Share of Trade (%)", fontsize=10, color=C_DKGRAY)
    ax.grid(axis="y", ls="--", alpha=0.25, color=C_GRAY)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(C_GRAY)
    ax.spines["bottom"].set_color(C_GRAY)

    top3_share = q5["share_2025"].head(3).sum()
    fig.text(
        0.50, 0.96,
        f"Trade remains concentrated — the top 3 provinces account for {top3_share:.1f}% of {YEAR2} flows",
        ha="center", fontsize=17, fontweight="bold", color=C_DKGREEN
    )
    sub_parts = [
        ("Rising concentration can improve scale efficiency", C_DKGREEN, "bold"),
        (" but increases regional dependency risk.", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, sub_parts, anchor=(0.50, 0.90), fontsize=11)
    return fig

def chart_q6_forecast():
    hist = monthly.copy()
    hist = hist[hist["Year"].isin([YEAR1, YEAR2])].copy()
    hist["Label"] = hist["Period"].dt.strftime("%b %Y")

    fc = forecast_monthly.copy()
    fc["Label"] = fc["Month"].dt.strftime("%b %Y")

    fig, ax = plt.subplots(figsize=(14, 7))
    fig.subplots_adjust(top=0.84, bottom=0.19, left=0.08, right=0.96)

    x_hist = np.arange(len(hist))
    x_fc = np.arange(len(hist), len(hist) + len(fc))

    ax.plot(x_hist, hist["total"] / 1e9, color=C_DKGREEN, lw=2.6, marker="o", ms=3.5, label=f"Historical ({YEAR1}-{YEAR2})")
    ax.plot(x_fc, fc["Forecast_Value"] / 1e9, color=C_RED, lw=2.4, marker="o", ms=3.5, ls="--", label=f"Baseline Outlook ({YEAR2 + 1})")

    if len(x_fc) > 0:
        ax.axvline(x_fc[0] - 0.5, color=C_DKGRAY, lw=1, ls="--", alpha=0.55)

    combined_labels = hist["Label"].tolist() + fc["Label"].tolist()
    tick_idx = list(range(0, len(combined_labels), 3))
    ax.set_xticks(tick_idx)
    ax.set_xticklabels([combined_labels[i] for i in tick_idx], rotation=30, ha="right", fontsize=8.5, color=C_DKGRAY)

    ax.set_ylabel("Total Trade (CA$ Billion)", fontsize=10, color=C_DKGRAY)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"CA${v:.1f}B"))
    ax.grid(axis="y", ls="--", alpha=0.25, color=C_GRAY)
    ax.legend(frameon=False, loc="upper left")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(C_GRAY)
    ax.spines["bottom"].set_color(C_GRAY)

    fig.text(
        0.50, 0.96,
        f"Baseline outlook suggests {YEAR2 + 1} trade near {fmt_cad(forecast_total_2026)} if current momentum holds",
        ha="center", fontsize=17, fontweight="bold", color=C_DKGREEN
    )
    sub_parts = [
        ("This is a directional planning baseline, not a causal forecast model.", C_DKGRAY, "normal"),
    ]
    add_text_subtitle(fig, sub_parts, anchor=(0.50, 0.90), fontsize=11)
    return fig

# =============================================================================
# 13. BUILD CHARTS
# =============================================================================
chart_registry = {
    "chart_q1_trade": chart_q1_annual_trade(),
    "chart_q1b_yoy": chart_q1b_monthly_yoy(),
    "chart_q2_province": chart_q2_province_growth_decline(),
    "chart_q3_balance": chart_q3_trade_balance(),
    "chart_q4_chapters": chart_q4_chapter_quadrant(),
    "chart_q5_concentration": chart_q5_concentration(),
    "chart_q6_forecast": chart_q6_forecast(),
}

chart_base64 = {}
for key, fig in chart_registry.items():
    save_fig(fig, key)
    chart_base64[key] = fig_to_base64(fig)
    plt.close(fig)

print("✅ Charts saved and converted to base64")

# =============================================================================
# 14. HTML TABLES
# =============================================================================
trade_stats_html = df_to_html_table(trade_stats_df, index=False)
prov_table_html = df_to_html_table(prov_table, index=False)
chapter_table_html = df_to_html_table(chapter_table, index=False)
forecast_html = df_to_html_table(forecast_table, index=False)
concentration_html = df_to_html_table(concentration_df, index=False)
expansion_html = df_to_html_table(expansion_df, index=False)

# =============================================================================
# 15. PREP TEXT FOR HTML
# =============================================================================
report_text_html = {}
for k, v in REPORT_TEXT.items():
    report_text_html[f"{k}_html"] = markdown_like_to_html(v)

# =============================================================================
# 16. HTML TEMPLATE
# =============================================================================
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>Canada-Brazil Trade Opportunities Report</title>
    <style>
        :root {
            --green-dark: #004D25;
            --green-light: #99CC33;
            --red: #E62310;
            --yellow: #FFCC22;
            --gray: #CCCCCC;
            --dark-gray: #555555;
            --rowalt: #F5F5F5;
            --bg: #F7FAF7;
        }

        * { box-sizing: border-box; }

        body {
            margin: 0;
            padding: 0;
            font-family: Arial, Helvetica, sans-serif;
            color: #222;
            background: #f4f7f4;
            line-height: 1.45;
            font-size: 10.5pt;
        }

        .page {
            max-width: 1120px;
            margin: 0 auto;
            background: white;
        }

        .cover {
            padding: 30px 34px 24px 34px;
            background: linear-gradient(180deg, #ffffff 0%, #f8fbf8 100%);
            border-bottom: 1px solid #e8efe8;
        }

        .eyebrow {
            color: var(--green-light);
            font-size: 10pt;
            font-weight: 700;
            letter-spacing: 0.06em;
            text-transform: uppercase;
            margin-bottom: 10px;
        }

        h1 {
            margin: 0 0 10px 0;
            color: var(--green-dark);
            font-size: 24pt;
            line-height: 1.15;
        }

        .subtitle {
            max-width: 860px;
            color: var(--dark-gray);
            font-size: 11pt;
            margin-bottom: 22px;
        }

        .kpi-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 14px;
            margin: 18px 0 18px 0;
        }

        .metric-card {
            border: 1px solid #e7ece7;
            border-radius: 12px;
            background: white;
            padding: 14px 14px 12px 14px;
        }

        .metric-card.red {
            border-color: #f0d4cf;
        }

        .metric-label {
            font-size: 9.5pt;
            color: var(--dark-gray);
            margin-bottom: 8px;
        }

        .metric-value {
            font-size: 18pt;
            color: var(--green-dark);
            font-weight: 700;
            margin-bottom: 4px;
            line-height: 1.1;
        }

        .metric-card.red .metric-value {
            color: var(--red);
        }

        .metric-sub {
            font-size: 9pt;
            color: var(--dark-gray);
        }

        .summary-box {
            background: #f8faf8;
            border-left: 4px solid var(--green-light);
            padding: 14px 16px;
            border-radius: 10px;
            margin: 14px 0;
        }

        .summary-box h3 {
            margin: 0 0 8px 0;
            color: var(--green-dark);
            font-size: 11pt;
        }

        .meta {
            color: #777;
            font-size: 8.8pt;
            margin-top: 10px;
        }

        .content {
            padding: 20px 34px 28px 34px;
        }

        .section {
            margin: 0 0 24px 0;
        }

        .section-header {
            color: var(--green-dark);
            font-size: 16pt;
            margin: 0 0 10px 0;
            padding-bottom: 4px;
            border-bottom: 2px solid #e8efe8;
            page-break-after: avoid;
        }

        .chart-container {
            margin: 12px 0 14px 0;
            padding: 10px 10px 6px 10px;
            border: 1px solid #e7ece7;
            border-radius: 10px;
            background: white;
        }

        .chart-container img {
            display: block;
            width: 100%;
            height: auto;
            border-radius: 6px;
        }

        .chart-title {
            font-size: 8.8pt;
            color: #666;
            margin-top: 6px;
        }

        .insight-box {
            background: #f8faf8;
            border-left: 4px solid var(--green-light);
            padding: 12px 14px;
            border-radius: 8px;
            margin: 10px 0 14px 0;
        }

        .note-box {
            background: #fffaf0;
            border-left: 4px solid var(--yellow);
            padding: 12px 14px;
            border-radius: 8px;
            margin: 10px 0 14px 0;
        }

        .table-container {
            margin: 12px 0 18px 0;
        }

        .table-container h4 {
            margin: 0 0 8px 0;
            color: var(--dark-gray);
            font-size: 10.5pt;
        }

        table.data-table {
            width: 100%;
            border-collapse: collapse;
            font-size: 8.8pt;
            table-layout: auto;
        }

        table.data-table th,
        table.data-table td {
            border: 1px solid #dddddd;
            padding: 6px 7px;
            vertical-align: top;
            word-break: break-word;
        }

        table.data-table th {
            background: #eef4ee;
            color: var(--green-dark);
            font-weight: 700;
        }

        table.data-table tr:nth-child(even) td {
            background: var(--rowalt);
        }

        p { margin: 0 0 8px 0; }
        ul { margin: 6px 0 6px 20px; }
        li { margin-bottom: 4px; }

        .footer-note {
            margin-top: 14px;
            font-size: 8.5pt;
            color: #777;
        }

        .chart-container,
        .table-container,
        .insight-box,
        .note-box,
        .summary-box,
        .metric-card {
            page-break-inside: avoid;
            break-inside: avoid;
        }

        @media print {
            body { font-size: 10pt; background: white; }
            .page { max-width: none; }
            table.data-table { font-size: 8.2pt; }
        }
    </style>
</head>
<body>
<div class="page">

    <div class="cover">
        <div class="eyebrow">Executive Trade Report</div>
        <h1>Canada-Brazil Trade Opportunities Report</h1>
        <div class="subtitle">
            Comparative review of merchandise trade performance in {{ YEAR1 }} and {{ YEAR2 }},
            with concentration analysis, product risk mapping, and a baseline outlook for {{ YEAR2 + 1 }}.
        </div>

        <div class="kpi-grid">
            <div class="metric-card">
                <div class="metric-label">Total Trade {{ YEAR2 }}</div>
                <div class="metric-value">{{ total_trade_2025 }}</div>
                <div class="metric-sub">{{ total_growth_pct }} vs {{ YEAR1 }}</div>
            </div>

            <div class="metric-card">
                <div class="metric-label">Exports {{ YEAR2 }}</div>
                <div class="metric-value">{{ exports_2025 }}</div>
                <div class="metric-sub">{{ exports_growth_pct }} vs {{ YEAR1 }}</div>
            </div>

            <div class="metric-card">
                <div class="metric-label">Imports {{ YEAR2 }}</div>
                <div class="metric-value">{{ imports_2025 }}</div>
                <div class="metric-sub">{{ imports_growth_pct }} vs {{ YEAR1 }}</div>
            </div>

            <div class="metric-card red">
                <div class="metric-label">Baseline Outlook {{ YEAR2 + 1 }}</div>
                <div class="metric-value">{{ forecast_total_2026 }}</div>
                <div class="metric-sub">{{ forecast_growth_pct }} implied growth baseline</div>
            </div>
        </div>

        <div class="summary-box">
            <h3>Automated Executive Summary</h3>
            <p>{{ executive_summary_auto }}</p>
        </div>

        <div class="summary-box">
            <h3>Your Commentary</h3>
            {{ report_text.executive_summary_html | safe }}
        </div>

        <div class="meta">Generated on {{ timestamp }}</div>
    </div>

    <div class="content">

        <div class="section">
            <h2 class="section-header">1. Trade Overview</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q1_trade }}">
                <div class="chart-title">Figure 1.1: Annual exports and imports comparison</div>
            </div>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q1b_yoy }}">
                <div class="chart-title">Figure 1.2: Monthly YoY performance comparison</div>
            </div>

            <div class="insight-box">
                {{ report_text.market_context_html | safe }}
                {{ report_text.trade_overview_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 1.1: Trade Summary Statistics</h4>
                {{ trade_stats_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">2. Provincial Growth & Decline</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q2_province }}">
                <div class="chart-title">Figure 2.1: Province growth and decline by YoY performance</div>
            </div>

            <div class="insight-box">
                {{ report_text.provincial_analysis_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 2.1: Provincial Performance Summary</h4>
                {{ prov_table_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">3. Trade Balance Dynamics</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q3_balance }}">
                <div class="chart-title">Figure 3.1: Monthly exports, imports, and net balance</div>
            </div>

            <div class="insight-box">
                {{ report_text.trade_balance_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">4. Product Winners vs Risks</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q4_chapters }}">
                <div class="chart-title">Figure 4.1: HS chapter winners vs risks quadrant</div>
            </div>

            <div class="insight-box">
                {{ report_text.product_analysis_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 4.1: Chapter Performance Summary</h4>
                {{ chapter_table_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">5. Concentration & Dependency</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q5_concentration }}">
                <div class="chart-title">Figure 5.1: Provincial concentration and cumulative trade share</div>
            </div>

            <div class="insight-box">
                {{ report_text.contribution_analysis_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 5.1: Concentration Summary</h4>
                {{ concentration_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">6. {{ YEAR2 + 1 }} Outlook</h2>

            <div class="chart-container">
                <img src="data:image/png;base64,{{ chart_q6_forecast }}">
                <div class="chart-title">Figure 6.1: Baseline forecast of total trade</div>
            </div>

            <div class="insight-box">
                {{ report_text.forecast_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 6.1: Monthly Forecast Values</h4>
                {{ forecast_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">7. Strategic Opportunities</h2>

            <div class="insight-box">
                {{ report_text.opportunities_html | safe }}
            </div>

            <div class="table-container">
                <h4>Table 7.1: Cross-Provincial Product Expansion Opportunities</h4>
                {{ expansion_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">8. Recommendations</h2>

            <div class="insight-box">
                {{ report_text.recommendations_html | safe }}
            </div>
        </div>

        <div class="section">
            <h2 class="section-header">9. Appendix / Technical Notes</h2>

            <div class="note-box">
                <p><strong>Automated technical summary:</strong></p>
                <ul>
                    <li>All values are in Canadian dollars (CAD), nominal terms.</li>
                    <li>Merchandise trade only; services are excluded.</li>
                    <li>Non-commercial chapters 97–99 were excluded when identifiable.</li>
                    <li>Product analysis uses a minimum baseline threshold of {{ min_baseline_label }}.</li>
                    <li>{{ YEAR2 + 1 }} forecast uses a seasonal distribution baseline with growth carried forward from {{ YEAR1 }} to {{ YEAR2 }}.</li>
                    <li>Generated on {{ timestamp }}.</li>
                </ul>
            </div>

            <div class="insight-box">
                {{ report_text.technical_notes_html | safe }}
            </div>

            <div class="footer-note">
                Final report prepared from dataset_clean.csv and supporting HS chapter mapping where available.
            </div>
        </div>

    </div>
</div>
</body>
</html>
"""

# =============================================================================
# 17. RENDER HTML
# =============================================================================
if not JINJA_AVAILABLE:
    raise ImportError("jinja2 is required to render the HTML report.")

template = Template(html_template)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

html_report = template.render(
    YEAR1=YEAR1,
    YEAR2=YEAR2,
    timestamp=timestamp,
    executive_summary_auto=executive_summary_auto,
    report_text=report_text_html,

    total_trade_2025=fmt_cad(total_2025),
    exports_2025=fmt_cad(exp_2025),
    imports_2025=fmt_cad(imp_2025),
    forecast_total_2026=fmt_cad(forecast_total_2026),

    total_growth_pct=fmt_pct(growth_pct),
    exports_growth_pct=fmt_pct(safe_pct_change(exp_2025, exp_2024)),
    imports_growth_pct=fmt_pct(safe_pct_change(imp_2025, imp_2024)),
    forecast_growth_pct=fmt_pct(forecast_growth),
    min_baseline_label=fmt_cad(MIN_BASELINE_THRESHOLD, d=0),

    trade_stats_html=trade_stats_html,
    prov_table_html=prov_table_html,
    chapter_table_html=chapter_table_html,
    forecast_html=forecast_html,
    concentration_html=concentration_html,
    expansion_html=expansion_html,

    **chart_base64
)

HTML_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
HTML_OUTPUT.write_text(html_report, encoding="utf-8")
print(f"✅ HTML report saved: {HTML_OUTPUT}")

# =============================================================================
# 18. EXPORT PDF
# =============================================================================
PDF_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

if WEASYPRINT_AVAILABLE:
    try:
        font_config = FontConfiguration()

        pdf_css = CSS(
            string="""
                @page {
                    size: A4;
                    margin: 1.55cm 1.25cm 1.55cm 1.25cm;
                }

                body {
                    font-family: Arial, Helvetica, sans-serif;
                    font-size: 10pt;
                    color: #222;
                    line-height: 1.40;
                }

                .cover, .chart-container, .table-container,
                .metric-card, .insight-box, .note-box, .summary-box {
                    page-break-inside: avoid;
                    break-inside: avoid;
                }

                .section-header {
                    font-size: 14pt;
                    margin-bottom: 8px;
                    page-break-after: avoid;
                }

                table.data-table {
                    font-size: 8pt;
                    table-layout: auto;
                }

                table.data-table th,
                table.data-table td {
                    padding: 5px 6px;
                    word-break: break-word;
                }

                .chart-title {
                    font-size: 8pt;
                }
            """,
            font_config=font_config
        )

        HTML(string=html_report).write_pdf(
            str(PDF_OUTPUT),
            stylesheets=[pdf_css],
            font_config=font_config
        )
        print(f"✅ PDF report saved: {PDF_OUTPUT}")

    except Exception as e:
        print(f"⚠️ WeasyPrint PDF export failed: {e}")
        try:
            fallback_html = REPORTS_DIR / "trade-report_fallback-open-this-in-browser.html"
            fallback_html.write_text(html_report, encoding="utf-8")
            print(f"⚠️ Fallback HTML saved: {fallback_html}")
        except Exception:
            pass
else:
    print("⚠️ WeasyPrint is not available. HTML was created, but PDF export was skipped.")

# =============================================================================
# 19. FINISH
# =============================================================================
print("\n" + "=" * 90)
print("REPORT COMPLETE")
print("=" * 90)
print(f"HTML       : {HTML_OUTPUT}")
print(f"PDF        : {PDF_OUTPUT}")
print(f"TEXT FILE  : {REPORT_TEXT_OUTPUT}")
print(f"IMAGE DIR  : {IMG_DIR}")
print("=" * 90)

Mounted at /content/drive
✅ Google Drive mounted
✅ Base folder   : /content/drive/MyDrive/canada-brazil-trade-report
✅ Reports folder: /content/drive/MyDrive/canada-brazil-trade-report/reports
✅ Notebook cwd  : /content
✅ REPORT_TEXT loaded: /content/drive/MyDrive/canada-brazil-trade-report/reports/report_text/trade-report_text.md
✅ Loading dataset: /content/drive/MyDrive/canada-brazil-trade-report/data/clean/dataset_clean.csv
✅ Loading HS map : /content/drive/MyDrive/business-report/archive/reference/hs_mapping.xlsx
📅 Comparing 2024 vs 2025


✅ Charts saved and converted to base64
✅ HTML report saved: /content/drive/MyDrive/canada-brazil-trade-report/reports/trade-report.html


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.003s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.006s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

✅ PDF report saved: /content/drive/MyDrive/canada-brazil-trade-report/reports/trade-report.pdf

REPORT COMPLETE
HTML       : /content/drive/MyDrive/canada-brazil-trade-report/reports/trade-report.html
PDF        : /content/drive/MyDrive/canada-brazil-trade-report/reports/trade-report.pdf
TEXT FILE  : /content/drive/MyDrive/canada-brazil-trade-report/reports/report_text/trade-report_text.md
IMAGE DIR  : /content/drive/MyDrive/canada-brazil-trade-report/reports/report_images
